# Ana Gomes · Databricks portfolio import

Run all cells in Databricks. This notebook includes the original synthetic fixture. The first code cell creates the project schema and managed input volume in the selected catalog. It rewrites only the project fixture files. Output tables are written by the original project pipeline.

Project source: https://github.com/anaoutro/crowdcanvas-databricks

Prepared for import on 5 October 2026. Execution in Databricks is pending; local syntax checks do not validate Spark/Delta runtime behavior.


In [ ]:
import re, json
default_catalog = spark.sql("SELECT current_catalog()").first()[0]
dbutils.widgets.text("catalog", default_catalog)
dbutils.widgets.text("schema", 'crowdcanvas')
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog), "Catalog must use a simple identifier"
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema_name), "Schema must use a simple identifier"
prefix = f"{catalog}.{schema_name}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {prefix}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {prefix}.input")
volume_root = f"/Volumes/{catalog}/{schema_name}/input"
fixture_csv = 'event_id,zone,event_time,arrival_time,occupancy,queue_wait_minutes\nOBS-FOOD-035,FOOD,2026-09-26T18:35:00Z,2026-09-26T18:40:00Z,232,14.25\nOBS-FOOD-013,FOOD,2026-09-26T18:13:00Z,2026-09-26T18:17:00Z,117,5.56\nOBS-HARBOR-015,HARBOR,2026-09-26T18:15:00Z,2026-09-26T18:20:00Z,183,4.32\nOBS-MAIN-008,MAIN,2026-09-26T18:08:00Z,2026-09-26T18:08:00Z,784,4.01\nOBS-HARBOR-003,HARBOR,2026-09-26T18:03:00Z,2026-09-26T18:08:00Z,207,5.66\nOBS-FOOD-007,FOOD,2026-09-26T18:07:00Z,2026-09-26T18:22:00Z,127,6.19\nOBS-HARBOR-031,HARBOR,2026-09-26T18:31:00Z,2026-09-26T18:36:00Z,427,17.71\nOBS-MAIN-037,MAIN,2026-09-26T18:37:00Z,2026-09-26T18:37:00Z,247,1.08\nOBS-GROVE-016,GROVE,2026-09-26T18:16:00Z,2026-09-26T18:21:00Z,322,4.85\nOBS-MAIN-073,MAIN,2026-09-26T19:13:00Z,2026-09-26T19:28:00Z,254,1.49\nOBS-FOOD-064,FOOD,2026-09-26T19:04:00Z,2026-09-26T19:09:00Z,229,13.92\nOBS-HARBOR-008,HARBOR,2026-09-26T18:08:00Z,2026-09-26T18:09:00Z,191,4.21\nOBS-HARBOR-058,HARBOR,2026-09-26T18:58:00Z,2026-09-26T19:00:00Z,452,18.26\nOBS-GROVE-009,GROVE,2026-09-26T18:09:00Z,2026-09-26T18:15:00Z,342,4.11\nOBS-FOOD-065,FOOD,2026-09-26T19:05:00Z,2026-09-26T19:07:00Z,227,13.17\nOBS-FOOD-021,FOOD,2026-09-26T18:21:00Z,2026-09-26T18:21:00Z,115,6.36\nOBS-FOOD-030,FOOD,2026-09-26T18:30:00Z,2026-09-26T18:33:00Z,236,14.76\nOBS-FOOD-011,FOOD,2026-09-26T18:11:00Z,2026-09-26T18:12:00Z,120,6.80\nOBS-HARBOR-050,HARBOR,2026-09-26T18:50:00Z,2026-09-26T18:56:00Z,454,17.07\nOBS-GROVE-035,GROVE,2026-09-26T18:35:00Z,2026-09-26T18:40:00Z,502,10.25\nOBS-FOOD-016,FOOD,2026-09-26T18:16:00Z,2026-09-26T18:16:00Z,121,6.99\nOBS-HARBOR-004,HARBOR,2026-09-26T18:04:00Z,2026-09-26T18:04:00Z,208,4.87\nOBS-HARBOR-025,HARBOR,2026-09-26T18:25:00Z,2026-09-26T18:30:00Z,181,5.08\nOBS-FOOD-015,FOOD,2026-09-26T18:15:00Z,2026-09-26T18:18:00Z,128,5.43\nOBS-FOOD-073,FOOD,2026-09-26T19:13:00Z,2026-09-26T19:28:00Z,232,13.47\nOBS-GROVE-004,GROVE,2026-09-26T18:04:00Z,2026-09-26T18:09:00Z,331,4.57\nOBS-HARBOR-066,HARBOR,2026-09-26T19:06:00Z,2026-09-26T19:06:00Z,431,18.18\nOBS-GROVE-044,GROVE,2026-09-26T18:44:00Z,2026-09-26T18:45:00Z,529,10.59\nOBS-GROVE-022,GROVE,2026-09-26T18:22:00Z,2026-09-26T18:25:00Z,304,5.90\nOBS-MAIN-002,MAIN,2026-09-26T18:02:00Z,2026-09-26T18:06:00Z,769,3.77\nOBS-GROVE-006,GROVE,2026-09-26T18:06:00Z,2026-09-26T18:21:00Z,338,4.44\nOBS-MAIN-040,MAIN,2026-09-26T18:40:00Z,2026-09-26T18:55:00Z,264,1.84\nOBS-FOOD-006,FOOD,2026-09-26T18:06:00Z,2026-09-26T18:10:00Z,128,5.30\nOBS-FOOD-071,FOOD,2026-09-26T19:11:00Z,2026-09-26T19:13:00Z,229,14.69\nOBS-GROVE-050,GROVE,2026-09-26T18:50:00Z,2026-09-26T19:05:00Z,514,10.69\nOBS-FOOD-025,FOOD,2026-09-26T18:25:00Z,2026-09-26T18:31:00Z,121,5.53\nOBS-MAIN-055,MAIN,2026-09-26T18:55:00Z,2026-09-26T18:59:00Z,271,1.41\nOBS-FOOD-055,FOOD,2026-09-26T18:55:00Z,2026-09-26T18:56:00Z,231,14.80\nOBS-HARBOR-056,HARBOR,2026-09-26T18:56:00Z,2026-09-26T18:58:00Z,430,17.74\nOBS-GROVE-054,GROVE,2026-09-26T18:54:00Z,2026-09-26T18:58:00Z,508,9.69\nOBS-MAIN-016,MAIN,2026-09-26T18:16:00Z,2026-09-26T18:18:00Z,735,3.89\nOBS-GROVE-031,GROVE,2026-09-26T18:31:00Z,2026-09-26T18:33:00Z,523,9.01\nOBS-GROVE-068,GROVE,2026-09-26T19:08:00Z,2026-09-26T19:10:00Z,491,9.99\nOBS-GROVE-028,GROVE,2026-09-26T18:28:00Z,2026-09-26T18:43:00Z,341,5.34\nOBS-GROVE-072,GROVE,2026-09-26T19:12:00Z,2026-09-26T19:27:00Z,511,9.13\nOBS-HARBOR-047,HARBOR,2026-09-26T18:47:00Z,2026-09-26T18:52:00Z,438,18.09\nOBS-FOOD-017,FOOD,2026-09-26T18:17:00Z,2026-09-26T18:19:00Z,121,5.34\nOBS-GROVE-020,GROVE,2026-09-26T18:20:00Z,2026-09-26T18:26:00Z,312,5.76\nOBS-GROVE-056,GROVE,2026-09-26T18:56:00Z,2026-09-26T18:56:00Z,495,10.86\nOBS-GROVE-073,GROVE,2026-09-26T19:13:00Z,2026-09-26T19:18:00Z,492,10.38\nOBS-HARBOR-041,HARBOR,2026-09-26T18:41:00Z,2026-09-26T18:44:00Z,454,17.52\nOBS-MAIN-039,MAIN,2026-09-26T18:39:00Z,2026-09-26T18:42:00Z,246,1.57\nOBS-FOOD-000,FOOD,2026-09-26T18:00:00Z,2026-09-26T18:03:00Z,126,5.49\nOBS-HARBOR-007,HARBOR,2026-09-26T18:07:00Z,2026-09-26T18:07:00Z,192,4.02\nOBS-GROVE-033,GROVE,2026-09-26T18:33:00Z,2026-09-26T18:39:00Z,506,9.71\nOBS-FOOD-061,FOOD,2026-09-26T19:01:00Z,2026-09-26T19:01:00Z,226,14.89\nOBS-MAIN-047,MAIN,2026-09-26T18:47:00Z,2026-09-26T18:48:00Z,242,2.70\nOBS-GROVE-000,GROVE,2026-09-26T18:00:00Z,2026-09-26T18:06:00Z,334,4.88\nOBS-FOOD-051,FOOD,2026-09-26T18:51:00Z,2026-09-26T19:06:00Z,234,13.16\nOBS-HARBOR-024,HARBOR,2026-09-26T18:24:00Z,2026-09-26T18:30:00Z,200,4.72\nOBS-MAIN-010,MAIN,2026-09-26T18:10:00Z,2026-09-26T18:13:00Z,779,4.62\nOBS-MAIN-033,MAIN,2026-09-26T18:33:00Z,2026-09-26T18:33:00Z,244,2.90\nOBS-FOOD-028,FOOD,2026-09-26T18:28:00Z,2026-09-26T18:31:00Z,121,6.79\nOBS-FOOD-033,FOOD,2026-09-26T18:33:00Z,2026-09-26T18:33:00Z,223,14.80\nOBS-FOOD-034,FOOD,2026-09-26T18:34:00Z,2026-09-26T18:34:00Z,223,14.05\nOBS-FOOD-003,FOOD,2026-09-26T18:03:00Z,2026-09-26T18:07:00Z,128,5.12\nOBS-MAIN-051,MAIN,2026-09-26T18:51:00Z,2026-09-26T19:06:00Z,265,2.76\nOBS-HARBOR-053,HARBOR,2026-09-26T18:53:00Z,2026-09-26T18:57:00Z,431,18.72\nOBS-MAIN-009,MAIN,2026-09-26T18:09:00Z,2026-09-26T18:10:00Z,754,3.57\nOBS-HARBOR-051,HARBOR,2026-09-26T18:51:00Z,2026-09-26T18:57:00Z,435,18.43\nOBS-HARBOR-001,HARBOR,2026-09-26T18:01:00Z,2026-09-26T18:02:00Z,209,4.43\nOBS-GROVE-037,GROVE,2026-09-26T18:37:00Z,2026-09-26T18:39:00Z,496,10.44\nOBS-FOOD-057,FOOD,2026-09-26T18:57:00Z,2026-09-26T19:02:00Z,240,14.50\nOBS-GROVE-071,GROVE,2026-09-26T19:11:00Z,2026-09-26T19:15:00Z,525,10.87\nOBS-FOOD-009,FOOD,2026-09-26T18:09:00Z,2026-09-26T18:09:00Z,122,6.80\nOBS-FOOD-040,FOOD,2026-09-26T18:40:00Z,2026-09-26T18:55:00Z,230,13.17\nOBS-HARBOR-005,HARBOR,2026-09-26T18:05:00Z,2026-09-26T18:20:00Z,193,5.93\nOBS-HARBOR-039,HARBOR,2026-09-26T18:39:00Z,2026-09-26T18:44:00Z,427,17.15\nOBS-FOOD-067,FOOD,2026-09-26T19:07:00Z,2026-09-26T19:10:00Z,234,14.80\nOBS-MAIN-056,MAIN,2026-09-26T18:56:00Z,2026-09-26T18:57:00Z,288,1.74\nOBS-GROVE-011,GROVE,2026-09-26T18:11:00Z,2026-09-26T18:14:00Z,304,4.32\nOBS-HARBOR-021,HARBOR,2026-09-26T18:21:00Z,2026-09-26T18:26:00Z,182,5.34\nOBS-FOOD-052,FOOD,2026-09-26T18:52:00Z,2026-09-26T18:55:00Z,234,14.67\nOBS-MAIN-021,MAIN,2026-09-26T18:21:00Z,2026-09-26T18:24:00Z,772,3.25\nOBS-HARBOR-060,HARBOR,2026-09-26T19:00:00Z,2026-09-26T19:15:00Z,450,17.72\nOBS-HARBOR-002,HARBOR,2026-09-26T18:02:00Z,2026-09-26T18:08:00Z,209,4.70\nOBS-HARBOR-022,HARBOR,2026-09-26T18:22:00Z,2026-09-26T18:25:00Z,196,5.31\nOBS-GROVE-065,GROVE,2026-09-26T19:05:00Z,2026-09-26T19:09:00Z,507,10.60\nOBS-MAIN-022,MAIN,2026-09-26T18:22:00Z,2026-09-26T18:26:00Z,763,3.48\nOBS-MAIN-058,MAIN,2026-09-26T18:58:00Z,2026-09-26T18:58:00Z,277,1.93\nOBS-FOOD-001,FOOD,2026-09-26T18:01:00Z,2026-09-26T18:07:00Z,129,5.90\nCONFLICT-1,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,701,4.94\nOBS-HARBOR-040,HARBOR,2026-09-26T18:40:00Z,2026-09-26T18:42:00Z,437,18.53\nOBS-GROVE-052,GROVE,2026-09-26T18:52:00Z,2026-09-26T18:55:00Z,500,10.35\nOBS-FOOD-053,FOOD,2026-09-26T18:53:00Z,2026-09-26T18:59:00Z,232,14.68\nOBS-HARBOR-059,HARBOR,2026-09-26T18:59:00Z,2026-09-26T19:02:00Z,450,17.02\nOBS-GROVE-001,GROVE,2026-09-26T18:01:00Z,2026-09-26T18:07:00Z,305,5.66\nOBS-GROVE-057,GROVE,2026-09-26T18:57:00Z,2026-09-26T18:59:00Z,512,9.66\nOBS-HARBOR-016,HARBOR,2026-09-26T18:16:00Z,2026-09-26T18:31:00Z,203,5.66\nOBS-MAIN-004,MAIN,2026-09-26T18:04:00Z,2026-09-26T18:07:00Z,756,4.07\nOBS-GROVE-000,GROVE,2026-09-26T18:00:00Z,2026-09-26T18:06:00Z,334,4.88\nOBS-GROVE-013,GROVE,2026-09-26T18:13:00Z,2026-09-26T18:18:00Z,338,5.96\nOBS-HARBOR-020,HARBOR,2026-09-26T18:20:00Z,2026-09-26T18:26:00Z,206,4.14\nOBS-MAIN-072,MAIN,2026-09-26T19:12:00Z,2026-09-26T19:18:00Z,279,2.70\nOBS-MAIN-025,MAIN,2026-09-26T18:25:00Z,2026-09-26T18:28:00Z,737,4.11\nOBS-FOOD-039,FOOD,2026-09-26T18:39:00Z,2026-09-26T18:41:00Z,229,14.32\nOBS-FOOD-054,FOOD,2026-09-26T18:54:00Z,2026-09-26T18:56:00Z,238,13.36\nOBS-MAIN-035,MAIN,2026-09-26T18:35:00Z,2026-09-26T18:38:00Z,297,2.92\nBAD-5,MAIN,2026-09-26T18:00:00Z,2026-09-26T17:00:00Z,765,4.94\nOBS-GROVE-002,GROVE,2026-09-26T18:02:00Z,2026-09-26T18:07:00Z,342,5.31\nOBS-MAIN-059,MAIN,2026-09-26T18:59:00Z,2026-09-26T19:04:00Z,266,1.57\nOBS-FOOD-029,FOOD,2026-09-26T18:29:00Z,2026-09-26T18:44:00Z,115,6.04\nOBS-FOOD-046,FOOD,2026-09-26T18:46:00Z,2026-09-26T18:52:00Z,227,14.80\nOBS-GROVE-005,GROVE,2026-09-26T18:05:00Z,2026-09-26T18:08:00Z,339,5.64\nOBS-MAIN-062,MAIN,2026-09-26T19:02:00Z,2026-09-26T19:17:00Z,288,2.09\nOBS-MAIN-001,MAIN,2026-09-26T18:01:00Z,2026-09-26T18:05:00Z,780,4.60\nOBS-HARBOR-035,HARBOR,2026-09-26T18:35:00Z,2026-09-26T18:35:00Z,441,17.59\nOBS-HARBOR-064,HARBOR,2026-09-26T19:04:00Z,2026-09-26T19:04:00Z,425,17.36\nOBS-HARBOR-014,HARBOR,2026-09-26T18:14:00Z,2026-09-26T18:19:00Z,194,5.94\nOBS-FOOD-026,FOOD,2026-09-26T18:26:00Z,2026-09-26T18:29:00Z,122,6.61\nOBS-MAIN-066,MAIN,2026-09-26T19:06:00Z,2026-09-26T19:06:00Z,285,2.99\nOBS-HARBOR-010,HARBOR,2026-09-26T18:10:00Z,2026-09-26T18:13:00Z,192,5.74\nOBS-MAIN-012,MAIN,2026-09-26T18:12:00Z,2026-09-26T18:18:00Z,749,4.12\nOBS-MAIN-018,MAIN,2026-09-26T18:18:00Z,2026-09-26T18:33:00Z,765,3.29\nOBS-MAIN-026,MAIN,2026-09-26T18:26:00Z,2026-09-26T18:29:00Z,733,3.81\nOBS-MAIN-069,MAIN,2026-09-26T19:09:00Z,2026-09-26T19:13:00Z,290,2.62\nOBS-GROVE-002,GROVE,2026-09-26T18:02:00Z,2026-09-26T18:07:00Z,342,5.31\nOBS-FOOD-018,FOOD,2026-09-26T18:18:00Z,2026-09-26T18:33:00Z,126,6.74\nOBS-HARBOR-002,HARBOR,2026-09-26T18:02:00Z,2026-09-26T18:08:00Z,209,4.70\nOBS-MAIN-074,MAIN,2026-09-26T19:14:00Z,2026-09-26T19:15:00Z,300,1.21\nOBS-FOOD-020,FOOD,2026-09-26T18:20:00Z,2026-09-26T18:26:00Z,121,5.94\nOBS-GROVE-034,GROVE,2026-09-26T18:34:00Z,2026-09-26T18:37:00Z,520,9.57\nOBS-GROVE-025,GROVE,2026-09-26T18:25:00Z,2026-09-26T18:27:00Z,311,5.12\nOBS-MAIN-064,MAIN,2026-09-26T19:04:00Z,2026-09-26T19:10:00Z,264,2.74\nOBS-GROVE-045,GROVE,2026-09-26T18:45:00Z,2026-09-26T18:47:00Z,496,9.99\nOBS-MAIN-049,MAIN,2026-09-26T18:49:00Z,2026-09-26T18:49:00Z,267,1.07\nOBS-HARBOR-070,HARBOR,2026-09-26T19:10:00Z,2026-09-26T19:16:00Z,426,18.00\nBAD-2,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,765,NaN\nOBS-GROVE-010,GROVE,2026-09-26T18:10:00Z,2026-09-26T18:15:00Z,315,5.94\nOBS-HARBOR-044,HARBOR,2026-09-26T18:44:00Z,2026-09-26T18:49:00Z,437,18.19\nOBS-FOOD-037,FOOD,2026-09-26T18:37:00Z,2026-09-26T18:43:00Z,235,13.78\nOBS-FOOD-050,FOOD,2026-09-26T18:50:00Z,2026-09-26T18:53:00Z,225,14.28\nOBS-GROVE-040,GROVE,2026-09-26T18:40:00Z,2026-09-26T18:43:00Z,531,10.33\nOBS-MAIN-024,MAIN,2026-09-26T18:24:00Z,2026-09-26T18:29:00Z,741,4.43\nOBS-GROVE-018,GROVE,2026-09-26T18:18:00Z,2026-09-26T18:21:00Z,314,4.97\nOBS-HARBOR-034,HARBOR,2026-09-26T18:34:00Z,2026-09-26T18:38:00Z,428,18.46\nOBS-HARBOR-001,HARBOR,2026-09-26T18:01:00Z,2026-09-26T18:02:00Z,209,4.43\nBAD-1,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,-1,4.94\nOBS-HARBOR-042,HARBOR,2026-09-26T18:42:00Z,2026-09-26T18:44:00Z,452,17.63\nOBS-FOOD-002,FOOD,2026-09-26T18:02:00Z,2026-09-26T18:08:00Z,123,5.86\nOBS-GROVE-058,GROVE,2026-09-26T18:58:00Z,2026-09-26T19:04:00Z,494,9.04\nOBS-HARBOR-030,HARBOR,2026-09-26T18:30:00Z,2026-09-26T18:35:00Z,426,18.34\nOBS-MAIN-013,MAIN,2026-09-26T18:13:00Z,2026-09-26T18:18:00Z,783,3.03\nOBS-GROVE-021,GROVE,2026-09-26T18:21:00Z,2026-09-26T18:24:00Z,304,5.79\nOBS-FOOD-058,FOOD,2026-09-26T18:58:00Z,2026-09-26T19:02:00Z,223,14.09\nOBS-HARBOR-006,HARBOR,2026-09-26T18:06:00Z,2026-09-26T18:11:00Z,202,5.42\nOBS-GROVE-059,GROVE,2026-09-26T18:59:00Z,2026-09-26T18:59:00Z,506,10.07\nOBS-MAIN-052,MAIN,2026-09-26T18:52:00Z,2026-09-26T18:57:00Z,295,2.30\nOBS-FOOD-059,FOOD,2026-09-26T18:59:00Z,2026-09-26T19:05:00Z,226,14.31\nOBS-MAIN-041,MAIN,2026-09-26T18:41:00Z,2026-09-26T18:43:00Z,243,2.52\nOBS-GROVE-026,GROVE,2026-09-26T18:26:00Z,2026-09-26T18:28:00Z,325,5.09\nOBS-FOOD-070,FOOD,2026-09-26T19:10:00Z,2026-09-26T19:14:00Z,239,14.75\nOBS-HARBOR-067,HARBOR,2026-09-26T19:07:00Z,2026-09-26T19:09:00Z,427,18.85\nOBS-HARBOR-018,HARBOR,2026-09-26T18:18:00Z,2026-09-26T18:22:00Z,203,4.91\nOBS-GROVE-003,GROVE,2026-09-26T18:03:00Z,2026-09-26T18:04:00Z,339,5.56\nOBS-FOOD-074,FOOD,2026-09-26T19:14:00Z,2026-09-26T19:20:00Z,232,14.30\nOBS-HARBOR-009,HARBOR,2026-09-26T18:09:00Z,2026-09-26T18:13:00Z,207,4.54\nOBS-FOOD-004,FOOD,2026-09-26T18:04:00Z,2026-09-26T18:10:00Z,126,6.54\nOBS-FOOD-019,FOOD,2026-09-26T18:19:00Z,2026-09-26T18:25:00Z,125,5.31\nOBS-MAIN-006,MAIN,2026-09-26T18:06:00Z,2026-09-26T18:08:00Z,734,3.83\nOBS-GROVE-060,GROVE,2026-09-26T19:00:00Z,2026-09-26T19:04:00Z,493,9.28\nOBS-FOOD-032,FOOD,2026-09-26T18:32:00Z,2026-09-26T18:38:00Z,226,13.40\nOBS-FOOD-044,FOOD,2026-09-26T18:44:00Z,2026-09-26T18:45:00Z,235,13.34\nOBS-MAIN-045,MAIN,2026-09-26T18:45:00Z,2026-09-26T18:48:00Z,278,1.67\nOBS-MAIN-067,MAIN,2026-09-26T19:07:00Z,2026-09-26T19:10:00Z,249,1.75\nOBS-FOOD-049,FOOD,2026-09-26T18:49:00Z,2026-09-26T18:52:00Z,225,14.61\nOBS-HARBOR-017,HARBOR,2026-09-26T18:17:00Z,2026-09-26T18:18:00Z,188,4.42\nOBS-GROVE-041,GROVE,2026-09-26T18:41:00Z,2026-09-26T18:45:00Z,517,10.69\nOBS-FOOD-002,FOOD,2026-09-26T18:02:00Z,2026-09-26T18:08:00Z,123,5.86\nOBS-GROVE-007,GROVE,2026-09-26T18:07:00Z,2026-09-26T18:09:00Z,339,4.32\nOBS-HARBOR-054,HARBOR,2026-09-26T18:54:00Z,2026-09-26T19:00:00Z,454,18.81\nOBS-HARBOR-019,HARBOR,2026-09-26T18:19:00Z,2026-09-26T18:20:00Z,190,4.29\nOBS-MAIN-070,MAIN,2026-09-26T19:10:00Z,2026-09-26T19:12:00Z,288,1.80\nOBS-MAIN-050,MAIN,2026-09-26T18:50:00Z,2026-09-26T18:56:00Z,278,2.55\nOBS-GROVE-008,GROVE,2026-09-26T18:08:00Z,2026-09-26T18:14:00Z,329,5.53\nOBS-MAIN-003,MAIN,2026-09-26T18:03:00Z,2026-09-26T18:05:00Z,741,4.34\nOBS-GROVE-064,GROVE,2026-09-26T19:04:00Z,2026-09-26T19:04:00Z,493,9.99\nOBS-GROVE-038,GROVE,2026-09-26T18:38:00Z,2026-09-26T18:40:00Z,502,9.53\nOBS-MAIN-017,MAIN,2026-09-26T18:17:00Z,2026-09-26T18:22:00Z,784,3.39\nOBS-GROVE-062,GROVE,2026-09-26T19:02:00Z,2026-09-26T19:08:00Z,502,9.91\nOBS-FOOD-068,FOOD,2026-09-26T19:08:00Z,2026-09-26T19:09:00Z,229,14.12\nOBS-GROVE-032,GROVE,2026-09-26T18:32:00Z,2026-09-26T18:35:00Z,507,10.89\nOBS-GROVE-015,GROVE,2026-09-26T18:15:00Z,2026-09-26T18:18:00Z,338,5.46\nOBS-HARBOR-032,HARBOR,2026-09-26T18:32:00Z,2026-09-26T18:35:00Z,432,17.42\nOBS-MAIN-048,MAIN,2026-09-26T18:48:00Z,2026-09-26T18:49:00Z,261,1.17\nOBS-HARBOR-029,HARBOR,2026-09-26T18:29:00Z,2026-09-26T18:29:00Z,208,5.65\nOBS-HARBOR-052,HARBOR,2026-09-26T18:52:00Z,2026-09-26T18:52:00Z,435,17.01\nOBS-FOOD-001,FOOD,2026-09-26T18:01:00Z,2026-09-26T18:07:00Z,129,5.90\nOBS-MAIN-057,MAIN,2026-09-26T18:57:00Z,2026-09-26T19:01:00Z,277,1.94\n,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,765,4.94\nOBS-MAIN-002,MAIN,2026-09-26T18:02:00Z,2026-09-26T18:06:00Z,769,3.77\nCONFLICT-1,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,700,4.94\nOBS-HARBOR-057,HARBOR,2026-09-26T18:57:00Z,2026-09-26T18:57:00Z,450,18.60\nOBS-MAIN-000,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,765,4.94\nOBS-MAIN-027,MAIN,2026-09-26T18:27:00Z,2026-09-26T18:29:00Z,776,4.62\nOBS-HARBOR-023,HARBOR,2026-09-26T18:23:00Z,2026-09-26T18:25:00Z,197,4.57\nOBS-FOOD-066,FOOD,2026-09-26T19:06:00Z,2026-09-26T19:11:00Z,232,14.29\nOBS-HARBOR-026,HARBOR,2026-09-26T18:26:00Z,2026-09-26T18:30:00Z,192,4.46\nOBS-FOOD-022,FOOD,2026-09-26T18:22:00Z,2026-09-26T18:27:00Z,113,6.93\nOBS-FOOD-000,FOOD,2026-09-26T18:00:00Z,2026-09-26T18:03:00Z,126,5.49\nOBS-MAIN-068,MAIN,2026-09-26T19:08:00Z,2026-09-26T19:08:00Z,273,2.38\nOBS-HARBOR-012,HARBOR,2026-09-26T18:12:00Z,2026-09-26T18:12:00Z,188,4.62\nOBS-GROVE-030,GROVE,2026-09-26T18:30:00Z,2026-09-26T18:31:00Z,500,10.87\nOBS-GROVE-066,GROVE,2026-09-26T19:06:00Z,2026-09-26T19:12:00Z,504,9.17\nOBS-MAIN-031,MAIN,2026-09-26T18:31:00Z,2026-09-26T18:36:00Z,243,2.70\nOBS-GROVE-049,GROVE,2026-09-26T18:49:00Z,2026-09-26T18:53:00Z,513,9.32\nOBS-HARBOR-045,HARBOR,2026-09-26T18:45:00Z,2026-09-26T18:46:00Z,440,17.95\nOBS-HARBOR-000,HARBOR,2026-09-26T18:00:00Z,2026-09-26T18:05:00Z,185,5.90\nOBS-GROVE-061,GROVE,2026-09-26T19:01:00Z,2026-09-26T19:16:00Z,530,9.85\nOBS-GROVE-029,GROVE,2026-09-26T18:29:00Z,2026-09-26T18:32:00Z,330,5.16\nOBS-HARBOR-071,HARBOR,2026-09-26T19:11:00Z,2026-09-26T19:26:00Z,439,17.89\nOBS-FOOD-005,FOOD,2026-09-26T18:05:00Z,2026-09-26T18:06:00Z,123,5.89\nOBS-FOOD-060,FOOD,2026-09-26T19:00:00Z,2026-09-26T19:05:00Z,230,13.31\nOBS-HARBOR-048,HARBOR,2026-09-26T18:48:00Z,2026-09-26T18:51:00Z,438,17.14\nOBS-FOOD-062,FOOD,2026-09-26T19:02:00Z,2026-09-26T19:17:00Z,230,13.73\nOBS-GROVE-047,GROVE,2026-09-26T18:47:00Z,2026-09-26T18:51:00Z,499,9.31\nOBS-GROVE-074,GROVE,2026-09-26T19:14:00Z,2026-09-26T19:14:00Z,493,10.04\nOBS-MAIN-028,MAIN,2026-09-26T18:28:00Z,2026-09-26T18:32:00Z,739,4.76\nOBS-HARBOR-038,HARBOR,2026-09-26T18:38:00Z,2026-09-26T18:53:00Z,432,17.10\nOBS-FOOD-012,FOOD,2026-09-26T18:12:00Z,2026-09-26T18:16:00Z,119,6.55\nOBS-MAIN-063,MAIN,2026-09-26T19:03:00Z,2026-09-26T19:03:00Z,275,2.81\nOBS-GROVE-042,GROVE,2026-09-26T18:42:00Z,2026-09-26T18:44:00Z,501,10.89\nOBS-FOOD-041,FOOD,2026-09-26T18:41:00Z,2026-09-26T18:45:00Z,230,13.74\nOBS-HARBOR-028,HARBOR,2026-09-26T18:28:00Z,2026-09-26T18:28:00Z,205,4.90\nOBS-HARBOR-046,HARBOR,2026-09-26T18:46:00Z,2026-09-26T18:46:00Z,428,18.65\nOBS-HARBOR-037,HARBOR,2026-09-26T18:37:00Z,2026-09-26T18:40:00Z,425,17.78\nOBS-GROVE-001,GROVE,2026-09-26T18:01:00Z,2026-09-26T18:07:00Z,305,5.66\nOBS-MAIN-014,MAIN,2026-09-26T18:14:00Z,2026-09-26T18:18:00Z,749,3.63\nOBS-FOOD-045,FOOD,2026-09-26T18:45:00Z,2026-09-26T18:45:00Z,227,13.30\nOBS-HARBOR-072,HARBOR,2026-09-26T19:12:00Z,2026-09-26T19:18:00Z,435,17.99\nOBS-MAIN-065,MAIN,2026-09-26T19:05:00Z,2026-09-26T19:07:00Z,270,1.41\nOBS-GROVE-036,GROVE,2026-09-26T18:36:00Z,2026-09-26T18:41:00Z,501,9.24\nOBS-GROVE-055,GROVE,2026-09-26T18:55:00Z,2026-09-26T18:59:00Z,519,9.82\nOBS-GROVE-046,GROVE,2026-09-26T18:46:00Z,2026-09-26T18:52:00Z,514,10.78\nOBS-MAIN-029,MAIN,2026-09-26T18:29:00Z,2026-09-26T18:44:00Z,756,4.25\nBAD-3,UNKNOWN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,765,4.94\nOBS-GROVE-070,GROVE,2026-09-26T19:10:00Z,2026-09-26T19:14:00Z,520,10.94\nOBS-HARBOR-069,HARBOR,2026-09-26T19:09:00Z,2026-09-26T19:11:00Z,446,18.22\nOBS-GROVE-014,GROVE,2026-09-26T18:14:00Z,2026-09-26T18:16:00Z,316,4.65\nOBS-GROVE-043,GROVE,2026-09-26T18:43:00Z,2026-09-26T18:44:00Z,519,10.64\nOBS-MAIN-011,MAIN,2026-09-26T18:11:00Z,2026-09-26T18:13:00Z,766,3.37\nOBS-GROVE-048,GROVE,2026-09-26T18:48:00Z,2026-09-26T18:51:00Z,520,10.22\nOBS-HARBOR-063,HARBOR,2026-09-26T19:03:00Z,2026-09-26T19:04:00Z,437,17.53\nOBS-GROVE-024,GROVE,2026-09-26T18:24:00Z,2026-09-26T18:28:00Z,320,5.89\nOBS-GROVE-023,GROVE,2026-09-26T18:23:00Z,2026-09-26T18:29:00Z,339,5.26\nBAD-4,MAIN,bad-date,2026-09-26T18:00:00Z,765,4.94\nOBS-MAIN-036,MAIN,2026-09-26T18:36:00Z,2026-09-26T18:36:00Z,241,2.93\nOBS-MAIN-071,MAIN,2026-09-26T19:11:00Z,2026-09-26T19:12:00Z,269,2.39\nOBS-HARBOR-000,HARBOR,2026-09-26T18:00:00Z,2026-09-26T18:05:00Z,185,5.90\nOBS-MAIN-005,MAIN,2026-09-26T18:05:00Z,2026-09-26T18:09:00Z,782,4.71\nOBS-MAIN-001,MAIN,2026-09-26T18:01:00Z,2026-09-26T18:05:00Z,780,4.60\nOBS-MAIN-030,MAIN,2026-09-26T18:30:00Z,2026-09-26T18:35:00Z,298,2.50\nOBS-HARBOR-036,HARBOR,2026-09-26T18:36:00Z,2026-09-26T18:39:00Z,437,17.77\nOBS-MAIN-015,MAIN,2026-09-26T18:15:00Z,2026-09-26T18:19:00Z,780,3.40\nOBS-MAIN-038,MAIN,2026-09-26T18:38:00Z,2026-09-26T18:42:00Z,249,2.92\nOBS-HARBOR-073,HARBOR,2026-09-26T19:13:00Z,2026-09-26T19:14:00Z,453,17.31\nOBS-MAIN-034,MAIN,2026-09-26T18:34:00Z,2026-09-26T18:37:00Z,267,1.94\nOBS-FOOD-010,FOOD,2026-09-26T18:10:00Z,2026-09-26T18:13:00Z,115,5.46\nOBS-FOOD-031,FOOD,2026-09-26T18:31:00Z,2026-09-26T18:35:00Z,237,14.46\nOBS-FOOD-038,FOOD,2026-09-26T18:38:00Z,2026-09-26T18:43:00Z,224,14.42\nOBS-GROVE-027,GROVE,2026-09-26T18:27:00Z,2026-09-26T18:30:00Z,313,4.35\nOBS-MAIN-046,MAIN,2026-09-26T18:46:00Z,2026-09-26T18:47:00Z,256,2.30\nOBS-FOOD-008,FOOD,2026-09-26T18:08:00Z,2026-09-26T18:09:00Z,121,5.84\nOBS-MAIN-032,MAIN,2026-09-26T18:32:00Z,2026-09-26T18:32:00Z,272,2.96\nOBS-FOOD-048,FOOD,2026-09-26T18:48:00Z,2026-09-26T18:53:00Z,228,13.36\nOBS-HARBOR-027,HARBOR,2026-09-26T18:27:00Z,2026-09-26T18:42:00Z,188,5.88\nOBS-FOOD-027,FOOD,2026-09-26T18:27:00Z,2026-09-26T18:27:00Z,118,6.75\nOBS-GROVE-019,GROVE,2026-09-26T18:19:00Z,2026-09-26T18:23:00Z,327,5.67\nOBS-FOOD-063,FOOD,2026-09-26T19:03:00Z,2026-09-26T19:07:00Z,238,13.41\nOBS-MAIN-054,MAIN,2026-09-26T18:54:00Z,2026-09-26T18:56:00Z,258,1.82\nOBS-MAIN-000,MAIN,2026-09-26T18:00:00Z,2026-09-26T18:00:00Z,765,4.94\nOBS-MAIN-019,MAIN,2026-09-26T18:19:00Z,2026-09-26T18:19:00Z,785,3.73\nOBS-FOOD-047,FOOD,2026-09-26T18:47:00Z,2026-09-26T18:53:00Z,234,14.08\nOBS-FOOD-023,FOOD,2026-09-26T18:23:00Z,2026-09-26T18:26:00Z,119,5.51\nOBS-GROVE-063,GROVE,2026-09-26T19:03:00Z,2026-09-26T19:09:00Z,512,10.11\nOBS-MAIN-053,MAIN,2026-09-26T18:53:00Z,2026-09-26T18:54:00Z,248,2.45\nOBS-GROVE-067,GROVE,2026-09-26T19:07:00Z,2026-09-26T19:11:00Z,492,9.53\nOBS-HARBOR-055,HARBOR,2026-09-26T18:55:00Z,2026-09-26T19:01:00Z,447,17.18\nOBS-FOOD-014,FOOD,2026-09-26T18:14:00Z,2026-09-26T18:19:00Z,114,6.37\nOBS-MAIN-061,MAIN,2026-09-26T19:01:00Z,2026-09-26T19:07:00Z,293,1.08\nOBS-FOOD-036,FOOD,2026-09-26T18:36:00Z,2026-09-26T18:40:00Z,227,14.89\nOBS-GROVE-069,GROVE,2026-09-26T19:09:00Z,2026-09-26T19:12:00Z,507,9.82\nOBS-FOOD-072,FOOD,2026-09-26T19:12:00Z,2026-09-26T19:17:00Z,233,13.27\nOBS-FOOD-069,FOOD,2026-09-26T19:09:00Z,2026-09-26T19:12:00Z,232,15.00\nOBS-GROVE-012,GROVE,2026-09-26T18:12:00Z,2026-09-26T18:13:00Z,338,4.25\nOBS-GROVE-017,GROVE,2026-09-26T18:17:00Z,2026-09-26T18:32:00Z,323,5.35\nOBS-HARBOR-065,HARBOR,2026-09-26T19:05:00Z,2026-09-26T19:08:00Z,441,18.98\nOBS-MAIN-042,MAIN,2026-09-26T18:42:00Z,2026-09-26T18:47:00Z,296,2.78\nOBS-MAIN-043,MAIN,2026-09-26T18:43:00Z,2026-09-26T18:45:00Z,243,1.06\nOBS-MAIN-020,MAIN,2026-09-26T18:20:00Z,2026-09-26T18:26:00Z,736,3.27\nOBS-GROVE-039,GROVE,2026-09-26T18:39:00Z,2026-09-26T18:54:00Z,505,10.51\nOBS-HARBOR-061,HARBOR,2026-09-26T19:01:00Z,2026-09-26T19:03:00Z,444,18.52\nOBS-HARBOR-049,HARBOR,2026-09-26T18:49:00Z,2026-09-26T19:04:00Z,444,18.18\nOBS-HARBOR-011,HARBOR,2026-09-26T18:11:00Z,2026-09-26T18:16:00Z,182,5.13\nOBS-GROVE-053,GROVE,2026-09-26T18:53:00Z,2026-09-26T18:55:00Z,524,10.96\nOBS-FOOD-056,FOOD,2026-09-26T18:56:00Z,2026-09-26T19:00:00Z,234,13.93\nOBS-MAIN-044,MAIN,2026-09-26T18:44:00Z,2026-09-26T18:50:00Z,289,2.42\nOBS-HARBOR-068,HARBOR,2026-09-26T19:08:00Z,2026-09-26T19:11:00Z,446,17.43\nOBS-HARBOR-043,HARBOR,2026-09-26T18:43:00Z,2026-09-26T18:47:00Z,453,18.46\nOBS-HARBOR-074,HARBOR,2026-09-26T19:14:00Z,2026-09-26T19:18:00Z,454,17.34\nOBS-MAIN-060,MAIN,2026-09-26T19:00:00Z,2026-09-26T19:04:00Z,276,2.81\nOBS-FOOD-043,FOOD,2026-09-26T18:43:00Z,2026-09-26T18:48:00Z,231,13.89\nOBS-HARBOR-013,HARBOR,2026-09-26T18:13:00Z,2026-09-26T18:19:00Z,205,5.69\nOBS-MAIN-023,MAIN,2026-09-26T18:23:00Z,2026-09-26T18:25:00Z,742,4.01\n\n'
default_input_path = volume_root + '/festival_events.csv'
dbutils.fs.put(default_input_path, fixture_csv, overwrite=True)
print("Synthetic fixture ready:", default_input_path)


# CrowdCanvas — Corrected historical festival flow
Synthetic, anonymous zone snapshots. Historical analysis retains all valid late arrivals.
Bronze incrementally merges payloads. Silver and Gold are rebuilt to revoke conflicted identities.
Upload data/festival_events.csv to a Unity Catalog volume before running.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
import re
dbutils.widgets.text("catalog",default_catalog)
dbutils.widgets.text("schema","crowdcanvas")
dbutils.widgets.text("input_path",default_input_path)
catalog=dbutils.widgets.get("catalog"); schema_name=dbutils.widgets.get("schema")
input_path=dbutils.widgets.get("input_path")
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*",catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*",schema_name)
assert input_path.startswith("/Volumes/") and input_path.endswith(".csv") and ".." not in input_path
prefix=f"{catalog}.{schema_name}"
spark.conf.set("spark.sql.session.timeZone","UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {prefix}")
columns=["event_id","zone","event_time","arrival_time","occupancy","queue_wait_minutes"]
zones=spark.createDataFrame([("MAIN",1000),("HARBOR",500),("GROVE",700),("FOOD",300)],["zone","capacity"])
schema=StructType([StructField(c,StringType(),True) for c in columns])


In [ ]:
raw=spark.read.option("header",True).option("mode","FAILFAST").schema(schema).csv(input_path)
incoming=(raw.withColumn("payload_hash",F.sha2(F.to_json(F.struct(*[F.col(c) for c in columns])),256))
    .withColumn("source_file",F.lit(input_path)).withColumn("ingested_at",F.current_timestamp()).dropDuplicates(["payload_hash"]))
bronze_table=f"{prefix}.bronze_snapshots"
if not spark.catalog.tableExists(bronze_table):
    incoming.write.format("delta").mode("append").saveAsTable(bronze_table)
else:
    (DeltaTable.forName(spark,bronze_table).alias("t").merge(incoming.alias("s"),"t.payload_hash=s.payload_hash")
     .whenNotMatchedInsertAll().execute())


In [ ]:
bronze=spark.table(bronze_table)
conflicted_ids=(bronze.filter(F.col("event_id").isNotNull() & (F.trim("event_id")!=""))
    .groupBy("event_id").agg(F.countDistinct("payload_hash").alias("versions"))
    .filter("versions > 1").select("event_id").withColumn("identity_conflict",F.lit(True)))
typed=(bronze.join(F.broadcast(zones),"zone","left")
    .join(conflicted_ids,"event_id","left")
    .withColumn("observed_at",F.expr("try_cast(event_time AS TIMESTAMP)"))
    .withColumn("arrived_at",F.expr("try_cast(arrival_time AS TIMESTAMP)"))
    .withColumn("occupancy_value",F.expr("try_cast(occupancy AS DECIMAL(10,2))"))
    .withColumn("wait_value",F.expr("try_cast(queue_wait_minutes AS DECIMAL(10,4))")))
missing=F.lit(False)
for c in columns: missing=missing|F.col(c).isNull()|(F.trim(F.col(c))=="")
bad_time=(F.col("observed_at").isNull()|F.col("arrived_at").isNull()|
    ~F.col("event_time").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:00Z$")|
    ~F.col("arrival_time").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:00Z$")|
    (F.col("arrived_at")<F.col("observed_at")))
occ=F.col("occupancy_value"); wait=F.col("wait_value")
bad_occ=occ.isNull()|(occ<0)|(occ>F.col("capacity"))|(occ!=F.floor(occ))|~F.col("occupancy").rlike(r"^\d{1,4}$")
bad_wait=wait.isNull()|(wait<0)|(wait>120)|(wait!=F.round(wait,2))|~F.col("queue_wait_minutes").rlike(r"^\d{1,3}(\.\d{1,2})?$")
typed=typed.withColumn("quality_reason",F.when(missing,"missing_field").when(F.col("capacity").isNull(),"unknown_zone")
    .when(bad_time,"invalid_timestamp").when(bad_occ,"invalid_occupancy").when(bad_wait,"invalid_wait")
    .when(F.col("identity_conflict"),"identity_conflict"))
quarantine=typed.filter(F.col("quality_reason").isNotNull())
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.quarantine_snapshots")
silver=(typed.filter(F.col("quality_reason").isNull()).select("event_id","zone","observed_at","arrived_at","capacity",
    F.col("occupancy_value").cast("int").alias("occupancy"),F.col("wait_value").cast("decimal(5,2)").alias("queue_wait_minutes")))
# Contract requires one unique snapshot per zone per minute; do not silently aggregate logical duplicates.
assert silver.count()==silver.select("zone","observed_at").distinct().count(), "Duplicate zone-minute snapshots"
assert silver.count()==silver.select("event_id").distinct().count(), "Duplicate accepted event IDs"
silver=(silver.withColumn("delay_seconds",F.unix_timestamp("arrived_at")-F.unix_timestamp("observed_at"))
    .withColumn("late_arrival",F.col("delay_seconds")>600))
silver.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.silver_snapshots")


In [ ]:
windows=(silver.groupBy(F.window("observed_at","5 minutes"),"zone","capacity")
    .agg(F.count("*").alias("samples"),F.max("occupancy").alias("peak_occupancy"),
         F.round(F.avg("queue_wait_minutes"),2).alias("mean_wait_minutes"),
         F.sum(F.col("late_arrival").cast("int")).alias("late_samples"))
    .select("zone","capacity",F.col("window.start").alias("window_start"),F.col("window.end").alias("window_end"),
            "samples","peak_occupancy","mean_wait_minutes","late_samples")
    .withColumn("peak_ratio",F.round(F.col("peak_occupancy")/F.col("capacity"),4))
    .withColumn("complete",F.col("samples")==5)
    .withColumn("pressure",(F.col("peak_ratio")>=.85)|(F.col("mean_wait_minutes")>=12)))
windows.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.gold_zone_windows")
cutoff=F.to_timestamp(F.lit("2026-09-26T18:30:00Z"))
summary=silver.groupBy("zone","capacity").agg(F.count("*").alias("samples"),F.max("occupancy").alias("peak_occupancy"),
    F.round(F.avg(F.when(F.col("observed_at")<cutoff,F.col("queue_wait_minutes"))),2).alias("before_wait"),
    F.round(F.avg(F.when(F.col("observed_at")>=cutoff,F.col("queue_wait_minutes"))),2).alias("after_wait"),
    F.round(F.avg(F.when(F.col("observed_at")<cutoff,F.col("occupancy"))),2).alias("before_mean_occupancy"),
    F.round(F.avg(F.when(F.col("observed_at")>=cutoff,F.col("occupancy"))),2).alias("after_mean_occupancy"))
summary.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{prefix}.gold_zone_summary")
assert windows.filter("samples > 5").count()==0
display(summary.orderBy("zone"));display(windows.filter("pressure OR NOT complete").orderBy("window_start","zone"))
display(quarantine.select("event_id","quality_reason"))


In [ ]:
assert spark.table(f"{prefix}.bronze_snapshots").count() == 302
assert spark.table(f"{prefix}.quarantine_snapshots").count() == 8
assert spark.table(f"{prefix}.silver_snapshots").count() == 294
assert spark.table(f"{prefix}.silver_snapshots").filter("late_arrival").count() == 27
assert spark.table(f"{prefix}.gold_zone_windows").count() == 60
assert spark.table(f"{prefix}.gold_zone_windows").filter("NOT complete").count() == 6
print("CROWDCANVAS HISTORY FIXTURE VALIDATED IN DATABRICKS")
